# Réseau de neurones simple avec PyTorch

Dans ce TP, nous allons construire deux réseaux de neurones avec PyTorch, en écrivant nous-mêmes leurs paramètres, leur perte et leur boucle d'entraînement : un réseau sans couche cachée d'abord, puis un perceptron multicouche. Le TP suivant construira les mêmes réseaux avec PyTorch Lightning, qui écrit la boucle à notre place.

Les données sont celles de MNIST, le jeu *Hello World!* du machine learning : des images de 28 $\times$ 28 pixels en niveaux de gris, qui représentent des chiffres de 0 à 9. Chaque image est associée à un label, le chiffre que l'image est censée représenter.

Les étapes :

1. Prendre en main les tenseurs PyTorch
2. Explorer et préparer les données, mettre de côté un ensemble de validation
3. Écrire un réseau sans couche cachée, sa perte, sa justesse et son `DataLoader`
4. L'entraîner par descente de gradient écrite à la main, puis avec un optimiseur PyTorch
5. Passer à un perceptron multicouche, le régulariser, choisir ses hyper-paramètres sur la validation
6. L'évaluer sur l'ensemble de test, le sauvegarder et le recharger

Ce notebook n'a pas besoin de GPU : chaque entraînement dure de quelques secondes à quelques minutes sur le processeur de Colab.

## Imports

In [ ]:
import random
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import seaborn
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Les calculs auront lieu sur le GPU s'il y en a un
device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch fournit les tenseurs, les gradients, les optimiseurs et le chargement des données par batchs ; torchvision, le jeu MNIST. [`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) et [`seaborn`](https://seaborn.pydata.org/) servent aux graphiques et aux opérations simples sur les tableaux.

## Prise en main des tenseurs

Sur Colab, PyTorch est déjà installé. Sur votre machine, installez-le avec `pip install torch torchvision`, ou avec la commande adaptée à votre système et à votre GPU que donne [pytorch.org](https://pytorch.org/get-started/locally/).

Avant de construire des réseaux, manipulons les tenseurs PyTorch, dont l'API est très proche de celle de NumPy (slides « NumPy et PyTorch » et « Calculer sur GPU »).

1. Créez un tenseur `a` de forme (2, 3) contenant les entiers de 0 à 5 (`torch.arange`, puis `reshape`). Quel est son `dtype` ? Convertissez-le en flottants (`.float()`), puis créez un tenseur `b` de forme (3,) rempli de 1 (`torch.ones`).
2. Calculez `a + b` et le produit matriciel `a @ b`. Quelles sont les formes des résultats ? Pourquoi `a + torch.ones(2)` échoue-t-il ? Une opération qui échoue lève une `RuntimeError` : attrapez-la avec `try`/`except` pour afficher son message.
3. Extrayez la deuxième colonne de `a`, puis sa dernière ligne. Ajoutez une dimension de taille 1 en tête de `a` avec `unsqueeze` : quelle forme obtient-on ? À quoi sert une telle dimension avec un modèle PyTorch ?
4. Convertissez `a` en tableau NumPy (`.numpy()`), puis de nouveau en tenseur (`torch.from_numpy`). Copiez `a` sur `device` (défini dans la cellule d'imports) et affichez son attribut `device`.

**Documentation utile**

- [`torch.arange`](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)
- [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- [`torch.Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unsqueeze.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.arange(6).reshape(2, 3)`, puis `a.float()`
- `a[:, 1]` : la deuxième colonne ; `a[-1]` : la dernière ligne
- `a.unsqueeze(0)` : une dimension de taille 1 en position 0
- `a.to(device)` : une copie sur `device` ; son attribut `device` dit où elle se trouve
- Une opération qui échoue, dans un `try` :

```python
try:
  a + torch.ones(2)
except RuntimeError as error:
  print(error)
```

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
a = torch.arange(...).reshape(...)
print(a.dtype)
a = a.float()
b = ...

print((a + b).shape, (a @ b).shape)
try:
  ...  # a + torch.ones(2)
except RuntimeError as error:
  print(error)

print(a[:, 1], ...)  # deuxième colonne, dernière ligne
print(a.unsqueeze(...).shape)

print(type(a.numpy()), torch.from_numpy(...).dtype)
print(a.to(...).device)
```

### Solution

In [ ]:
a = torch.arange(6).reshape(2, 3)
print(a, a.dtype)
a = a.float()
b = torch.ones(3)

# b est diffusé (broadcasting) sur chaque ligne de a
print(a + b, (a + b).shape)
# Produit matrice-vecteur : la somme de chaque ligne de a
print(a @ b, (a @ b).shape)
try:
  a + torch.ones(2)
except RuntimeError as error:
  print("a + torch.ones(2) :", error)

print(a[:, 1])  # deuxième colonne
print(a[-1])  # dernière ligne
print(a.unsqueeze(0).shape)  # dimension de batch de taille 1

a_numpy = a.numpy()
print(type(a_numpy), torch.from_numpy(a_numpy).dtype)
print(a.to(device).device)

- `torch.arange` crée des entiers 64 bits (`torch.int64`) ; les réseaux de neurones calculent en flottants, par défaut `torch.float32`.
- `a + b` a la forme (2, 3) et `a @ b` la forme (2,). Pour le broadcasting, les dimensions sont comparées en partant de la dernière : elles doivent être égales, ou l'une d'elles doit valoir 1 ou être absente. (2, 3) et (3,) sont compatibles, mais pas (2, 3) et (2,) : il faudrait la forme (2, 1), `torch.ones(2).unsqueeze(1)`.
- `a.unsqueeze(0)` a la forme (1, 2, 3). Les modèles PyTorch traitent des batchs, la première dimension étant celle des exemples : pour prédire sur un seul exemple, on lui ajoute une dimension de batch de taille 1.
- `a.numpy()` partage la mémoire du tenseur, sans copie : il faut pour cela un tenseur sur CPU (`.cpu()`) et hors du graphe d'autograd (`.detach()`). `torch.from_numpy` conserve le type NumPy (`float32` ici, mais `float64` pour un tableau NumPy créé par défaut).
- Sans GPU, `a.to(device).device` vaut `cpu` ; avec un GPU, `cuda:0`.

## Récupération des données

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Regardons les données

Utilisez la cellule suivante pour explorer les tenseurs `X_train`, `y_train`, `X_test` et `y_test` : leur forme (`shape`), leur type (`dtype`), leurs valeurs extrêmes, et quelques images avec `plt.imshow`.

Vous essayerez de répondre aux questions suivantes :

- Combien y a-t-il d'images au total dans ce dataset ?
- Sous quelle forme sont stockées les images ? les labels ?
- Les classes sont-elles équilibrées ?

**Documentation utile**

- [`torch.bincount`](https://docs.pytorch.org/docs/stable/generated/torch.bincount.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()`, `X_train.max()` : forme, type et valeurs extrêmes
- `torch.bincount(y_train)` : le nombre d'exemples de chaque classe, de 0 à 9
- `plt.imshow(X_train[0], cmap="gray_r")` : la première image, en noir sur blanc

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, ...)
print(X_test.shape, ...)

print(torch.bincount(...))

plt.imshow(..., cmap="gray_r")
plt.title(f"Label : {int(y_train[0])}")
plt.show()
```

### Solution

In [ ]:
print(f"Images d'entraînement : {tuple(X_train.shape)}, {X_train.dtype}")
print(f"Valeurs des pixels : de {int(X_train.min())} à {int(X_train.max())}")
print(f"Labels d'entraînement : {tuple(y_train.shape)}, {y_train.dtype}")
print(f"Images et labels de test : {tuple(X_test.shape)}, {tuple(y_test.shape)}")
print(f"Exemples par classe : {torch.bincount(y_train).tolist()}")

plt.imshow(X_train[0], cmap="gray_r")
plt.title(f"Premier exemple du dataset ({int(y_train[0])})")
plt.show()

seaborn.countplot(x=y_train.numpy())
plt.title("Décompte des différentes classes (chiffres)")
plt.ylabel("Décompte")
plt.xlabel("Chiffre")
plt.show()

- 70 000 images : 60 000 d'entraînement et 10 000 de test.
- Les images sont un tenseur `torch.uint8` de forme (60 000, 28, 28) : un entier de 0 (fond) à 255 (encre) par pixel. Les labels sont un tenseur `torch.int64` de forme (60 000,), un chiffre de 0 à 9 par image.
- Les classes sont à peu près équilibrées : de 5 421 exemples (le 5) à 6 742 (le 1). La justesse, la proportion de bonnes réponses, est donc une métrique pertinente.

## Affichage d'exemples

Affichez 25 exemples, tirés au hasard dans la base d'entraînement, sur une grille de 5 $\times$ 5. Affichez le label de chacun dans son titre.

**Documentation utile**

- [`numpy.random.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.choice.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# Une grille de 5 x 5 sous-graphiques
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.random.choice(len(X_train), size=(5, 5), replace=False)` : 25 indices distincts, rangés en grille
- `ax[i, j].imshow(image, cmap="gray_r")`, `ax[i, j].set_title(...)` et `ax[i, j].axis("off")` : l'image, son titre et sans axes
- `int(y_train[index])` : le label, en entier Python plutôt qu'en tenseur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
f, ax = plt.subplots(5, 5, figsize=(15, 15))
indices = np.random.choice(..., size=(5, 5), replace=False)
for i in range(5):
  for j in range(5):
    index = indices[i, j]
    ax[i, j].imshow(..., cmap="gray_r")
    ax[i, j].set_title(f"Exemple {index} ({...})")
    ax[i, j].axis("off")
```

### Solution

In [ ]:
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# 25 indices au hasard, sans remise (on ne veut pas afficher la même image deux
# fois)
indices = np.random.choice(len(X_train), size=(5, 5), replace=False)

for i in range(5):
  for j in range(5):
    index = indices[i, j]
    ax[i, j].imshow(X_train[index], cmap="gray_r")
    ax[i, j].set_title(f"Exemple {index} ({int(y_train[index])})")
    ax[i, j].axis("off")

## Transformation des données

Trois transformations préparent les données pour le réseau :

1. Mettre les images à plat : un réseau sans couche cachée prend un vecteur par exemple, de forme `(_, 28²)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt qu'entre $0$ et $255$.
3. Copier les images et les labels sur `device`, où le modèle calculera (slide « Calculer sur GPU »).

*Appliquez ces trois transformations à `X_train` et à `X_test`, et la troisième à `y_train` et à `y_test`.*

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X.reshape(-1, 28 * 28)` : `-1` laisse PyTorch calculer le nombre de lignes
- `X / 255` : des valeurs entre 0 et 1 ; la division d'un tenseur d'entiers donne des `torch.float32`
- `X.to(device)` : la copie sur `device`

Attendu : `X_train.shape` vaut `torch.Size([60000, 784])`, `X_train.dtype` vaut `torch.float32` et `X_train.max()` vaut 1.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = (X_train.reshape(-1, ...) / ...).to(device)
X_test = ...
y_train, y_test = y_train.to(...), ...
print(X_train.shape, X_train.dtype, X_train.max())
```

### Solution

In [ ]:
input_dim = 28 * 28

# Mise à plat : un vecteur de input_dim pixels par image, puis passage dans
# [0, 1] et copie sur le périphérique de calcul
X_train = (X_train.reshape(-1, input_dim) / 255).to(device)
X_test = (X_test.reshape(-1, input_dim) / 255).to(device)
y_train, y_test = y_train.to(device), y_test.to(device)
print(X_train.shape, X_train.dtype, X_train.max())

## Séparation entraînement / validation

Pour choisir les hyper-paramètres (learning rate, taille du réseau…), on compare des modèles sur un ensemble de **validation**. L'ensemble de test est réservé à l'évaluation finale : s'en servir pour faire des choix rendrait cette évaluation trop optimiste.

On met de côté les 10 000 derniers exemples d'entraînement pour la validation :

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Réseau de neurones sans couche cachée

Un réseau de neurones sans couche cachée calcule $\text{softmax}(XW + b)$ : c'est la **régression logistique multinomiale**. `W` est la matrice des poids, multipliée à l'entrée, et `b` le vecteur des biais, ajouté ensuite. La softmax transforme les 10 scores de chaque exemple en probabilités.

### Initialisation des paramètres

*Complétez la fonction `create_simple_nn_weights(n_inputs, n_outputs)`, qui renvoie deux [paramètres PyTorch](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) créés sur `device` : `W`, de forme `(n_inputs, n_outputs)`, tiré d'une loi normale (`torch.randn`) et divisé par $\sqrt{n_\text{inputs}}$ ; `b`, un vecteur de zéros de forme `(n_outputs,)`.*

Cette division donne aux poids une variance de $1 / n_\text{inputs}$ : la sortie de la couche, une somme de $n_\text{inputs}$ termes, reste de l'ordre de grandeur de ses entrées, quel que soit leur nombre (slide « Initialisation des poids »).

**Documentation utile**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.zeros`](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

In [ ]:
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  pass  # Votre code ici

#### Conseils supplémentaires

- `torch.randn(n_inputs, n_outputs, device=device)` : un tenseur de moyenne 0 et d'écart type 1, créé directement sur `device`
- `n_inputs**0.5` : la racine carrée de `n_inputs`
- `torch.zeros(n_outputs, device=device)` : les biais, nuls au départ
- `nn.Parameter(tenseur)` : un tenseur dont PyTorch calcule le gradient, et qu'un `nn.Module` reconnaît comme paramètre

Attendu : `W` a la forme (784, 10) et un écart type (`W.detach().std()`) proche de $1 / 28 \approx 0{,}036$.

```python
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  W = torch.randn(..., ..., device=device) / ...
  b = torch.zeros(..., device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = create_simple_nn_weights(784, 10)
print(W.shape, b.shape, W.detach().std())
```

#### Solution

In [ ]:
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  # Variance 1 / n_inputs : des sorties de l'ordre de 1, quel que soit n_inputs
  W = torch.randn(n_inputs, n_outputs, device=device) / n_inputs**0.5
  b = torch.zeros(n_outputs, device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = create_simple_nn_weights(784, 10)
print(W.shape, b.shape, W.detach().std())

### Définition du modèle

*Complétez la méthode `forward` de `SimpleNN`, qui renvoie le logarithme de $\text{softmax}(XW + b)$ : la fonction `torch.log_softmax`, sur la dernière dimension, celle des classes.*

Le logarithme de la softmax, calculé en une fois, est plus stable numériquement que la softmax suivie d'un logarithme. Ces log-probabilités sont ce qu'attend la fonction de perte de la partie suivante.

**Documentation utile**

- [`torch.log_softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Votre code ici

#### Conseils supplémentaires

- `X @ self.W + self.b` : le produit matriciel, puis l'ajout des biais, diffusé sur chaque ligne
- `torch.log_softmax(..., dim=-1)` : la softmax de chaque ligne, puis son logarithme

Attendu : `SimpleNN()(X_train[:5])` a la forme (5, 10), et l'exponentielle de chaque ligne a une somme de 1 (`.exp().sum(dim=-1)`).

```python
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    return torch.log_softmax(..., dim=...)
```

#### Solution

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Des log-probabilités : le logarithme de la softmax de chaque ligne
    return torch.log_softmax(X @ self.W + self.b, dim=-1)


y_pred = SimpleNN()(X_train[:5])
print(y_pred.shape, y_pred.exp().sum(dim=-1))

### Calcul de la fonction de perte

La fonction de perte standard en classification multi-classes est l'entropie croisée : la moyenne, sur les exemples, de l'opposé de la log-probabilité que le modèle donne à la bonne classe. Notre modèle renvoyant déjà des log-probabilités, `torch.nn.functional.nll_loss` la calcule : elle prend les log-probabilités d'abord, puis les labels entiers (0 à 9), et renvoie leur moyenne.

*Complétez la fonction `categorical_crossentropy`. Quelle valeur prend-elle pour un modèle qui n'a pas encore appris ?*

**Documentation utile**

- [`torch.nn.functional.nll_loss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.nll_loss.html)

In [ ]:
def categorical_crossentropy(y: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
  pass  # Votre code ici

#### Conseils supplémentaires

- `nn.functional.nll_loss(y_pred, y)` : attention à l'ordre, les prédictions d'abord ; elle renvoie la moyenne sur les exemples
- `nll_loss` sur des log-probabilités calcule la même chose que `nn.functional.cross_entropy` sur les scores bruts : cette dernière applique `log_softmax` elle-même (slide « Piège classique : softmax et entropie croisée »)

Un modèle qui n'a pas appris donne à peu près la même probabilité, 1/10, à chaque classe : sa perte vaut environ $-\log(1/10) = \log 10 \approx 2{,}3$.

```python
def categorical_crossentropy(y: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
  return nn.functional.nll_loss(..., ...)


print(categorical_crossentropy(y_train, SimpleNN()(X_train)))
```

#### Solution

In [ ]:
def categorical_crossentropy(y: torch.Tensor, y_pred: torch.Tensor) -> torch.Tensor:
  # Moyenne sur les exemples de -log(probabilité de la bonne classe)
  return nn.functional.nll_loss(y_pred, y)


# Environ log(10) = 2,3 : le modèle répond presque au hasard
print(categorical_crossentropy(y_train, SimpleNN()(X_train)))

### Métriques

Pour savoir si un modèle apprend, on mesure ses performances. Nous utiliserons la plus simple, mais aussi une des plus informatives : la justesse (*accuracy*), le nombre de bonnes prédictions sur le nombre total de prédictions.

*Complétez la fonction `accuracy`. La classe prédite est celle de la plus grande log-probabilité (`torch.argmax`) : comparez-la aux labels, puis calculez la proportion de bonnes réponses.*

**Documentation utile**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  pass  # Votre code ici

#### Conseils supplémentaires

- `torch.argmax(y_pred, dim=-1)` : la classe prédite pour chaque exemple
- `predictions == y` : un tenseur de booléens
- `.float().mean()` : la proportion de `True`
- `.item()` : un nombre Python, plus simple à afficher et à stocker

Attendu : environ 0,1 pour un modèle qui n'a pas appris, une chance sur dix.

```python
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(..., dim=...)
  return (... == ...).float().mean().item()


print(accuracy(y_val, SimpleNN()(X_val)))
```

#### Solution

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(y_pred, dim=-1)
  return (predictions == y).float().mean().item()


print(accuracy(y_val, SimpleNN()(X_val)))

## Création d'un `DataLoader` PyTorch

Le modèle apprendra par mini-batchs : à chaque pas, un petit groupe d'exemples tirés au hasard. Un `DataLoader` les prépare (slides « Charger ses données : `DataLoader` » et « Boucle d'entraînement complète »).

*Complétez la fonction `create_dataset`, qui enveloppe les tenseurs `X` et `y` dans un `TensorDataset`, puis dans un `DataLoader` qui renvoie des batchs `(X_batch, y_batch)` de `batch_size` exemples, mélangés à chaque epoch.*

In [ ]:
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  pass  # Votre code ici

### Conseils supplémentaires

- `TensorDataset(X, y)` : un dataset dont l'exemple `i` est le couple `(X[i], y[i])`
- `DataLoader(dataset, batch_size=batch_size, shuffle=True)` : des batchs, dans un ordre tiré à nouveau à chaque parcours

Attendu : `create_dataset(X_train, y_train)` compte 391 batchs (`len`), et le premier, `next(iter(...))`, a des images de forme (128, 784).

```python
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  return DataLoader(TensorDataset(..., ...), batch_size=..., shuffle=...)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)
```

### Solution

In [ ]:
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  # Les exemples sont mélangés à chaque epoch
  return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)

## Apprentissage

Voici une boucle d'apprentissage presque complète : elle parcourt les batchs à chaque epoch, puis mesure la perte et la justesse sur les ensembles d'entraînement et de validation, et trace leurs courbes à la fin.

*Implémentez la partie manquante : la mise à jour des paramètres du modèle pour un batch, par descente de gradient. Calculez la perte du batch, en lui ajoutant `regularization(model)` quand une régularisation est donnée (elle servira plus loin), remettez les gradients à zéro, calculez-les avec `backward`, puis retirez de chaque paramètre `learning_rate` fois son gradient (slides « Différentiation automatique » et « Désactiver le suivi des gradients »).*

In [ ]:
LossFunction = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
RegularizationFunction = Callable[[nn.Module], torch.Tensor]


def train(
  model: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Métriques initiales : acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Votre code ici

    # Calcul et affichage des métriques toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {e + 1} : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Conseils supplémentaires

- `loss(y_batch, model(X_batch))` : la perte du batch ; on lui ajoute `regularization(model)` si `regularization is not None`
- `model.zero_grad()` : remet à zéro les gradients, qui s'accumulent d'un `backward` à l'autre
- `loss_value.backward()` : range le gradient de la perte dans l'attribut `.grad` de chaque paramètre
- `with torch.no_grad():` : la mise à jour ne doit pas entrer dans le graphe de calcul
- `parameter -= learning_rate * parameter.grad` : la mise à jour, en place, pour chaque paramètre de `model.parameters()`

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_batch, y_batch in dataset_train:
  loss_value = ...
  if regularization is not None:
    loss_value = loss_value + ...
  model.zero_grad()
  loss_value.backward()
  with torch.no_grad():
    for parameter in model.parameters():
      parameter -= ...
```

### Solution

In [ ]:
LossFunction = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
RegularizationFunction = Callable[[nn.Module], torch.Tensor]


def train(
  model: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train, y_train_pred))]
    val_losses = [float(loss(y_val, y_val_pred))]

  print(
    f"Métriques initiales : acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      # Calcul de la perte, avec la pénalité de régularisation s'il y en a une
      loss_value = loss(y_batch, model(X_batch))
      if regularization is not None:
        loss_value = loss_value + regularization(model)

      # Calcul automatique du gradient de la perte par rapport à chaque
      # paramètre : il est rangé dans l'attribut .grad du paramètre
      model.zero_grad()
      loss_value.backward()

      # Mise à jour de chaque paramètre selon la règle :
      # nouvelle_valeur = ancienne_valeur - learning_rate * gradient
      with torch.no_grad():
        for parameter in model.parameters():
          parameter -= learning_rate * parameter.grad

    # Calcul et affichage des métriques toutes les `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train, y_train_pred)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val, y_val_pred)))
      print(
        f"Epoch {e + 1} : acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Entraînement")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Fonction de perte pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Entraînement")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Justesse pendant l'entraînement")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Entraînement du réseau sans couche cachée

10 epochs, avec un learning rate de 0,5 :

In [ ]:
simple_nn = SimpleNN()
_ = train(simple_nn, epochs=10, learning_rate=0.5)

*Quelle justesse de validation atteint ce modèle ? Progresse-t-il encore après quelques epochs ?*

Votre réponse ici

#### Solution

Environ 92,5 % de justesse en validation. L'essentiel est appris dès la première epoch (environ 91,5 %) ; ensuite, le modèle ne gagne plus qu'un point en 9 epochs, en oscillant d'une epoch à l'autre. Les justesses d'entraînement et de validation restent égales : le modèle ne sur-apprend pas, il est trop simple. Une régression logistique sépare les classes par des frontières linéaires, ce qui plafonne vers 92 % sur MNIST.

## Avec un optimiseur PyTorch

La mise à jour écrite à la main ci-dessus (`parameter -= learning_rate * parameter.grad`) est exactement ce que fait l'optimiseur `torch.optim.SGD`. En pratique, on écrit donc la boucle d'entraînement standard (slide « Boucle d'entraînement complète ») :

```python
optimizer.zero_grad()  # remise à zéro des gradients du batch précédent
loss.backward()  # calcul des gradients
optimizer.step()  # mise à jour des paramètres
```

*Complétez la boucle de `train_with_optimizer`, qui fait un pas de `optimizer` par batch. La cellule qui suit la solution entraîne un `SimpleNN` avec `torch.optim.SGD` et le même learning rate que précédemment, puis un autre avec `torch.optim.Adam`.*

**Documentation utile**

- [`torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

In [ ]:
def train_with_optimizer(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Votre code ici

    with torch.no_grad():
      val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1} : val_acc {val_accuracy:.4f}")

### Conseils supplémentaires

- La perte se calcule comme dans `train`, avec `categorical_crossentropy`, sans régularisation
- `optimizer.zero_grad()`, `loss_value.backward()`, puis `optimizer.step()` : un pas de l'optimiseur sur tous les paramètres qu'on lui a donnés à sa création
- `torch.optim.SGD(model.parameters(), lr=0.5)` : un optimiseur est lié aux paramètres d'un modèle ; un nouveau modèle demande un nouvel optimiseur

La boucle sur les batchs complétée, à copier à la place de la sienne (avec son indentation) :

```python
for X_batch, y_batch in dataset_train:
  optimizer.zero_grad()
  loss_value = categorical_crossentropy(..., ...)
  ...  # calcul des gradients
  ...  # mise à jour des paramètres
```

### Solution

In [ ]:
def train_with_optimizer(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      optimizer.zero_grad()
      loss_value = categorical_crossentropy(y_batch, model(X_batch))
      loss_value.backward()
      optimizer.step()

    with torch.no_grad():
      val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1} : val_acc {val_accuracy:.4f}")

### Comparaison de SGD et d'Adam

5 epochs avec chacun : SGD avec le learning rate de la boucle écrite à la main, Adam avec son learning rate usuel, bien plus petit.

In [ ]:
print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=0.5))

print("Adam")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.Adam(simple_nn.parameters(), lr=1e-3))

*Obtenez-vous avec SGD les performances de la boucle écrite à la main ? Que change Adam ?*

Votre réponse ici

#### Solution

Avec SGD et le même learning rate, on retrouve les performances de la boucle écrite à la main (environ 92 % en 5 epochs) : `optimizer.step()` applique la même règle de mise à jour, et `optimizer.zero_grad()` remplace `model.zero_grad()`. Changer d'algorithme d'optimisation ne demande plus que de changer d'optimiseur. Adam adapte le pas de chaque paramètre à l'historique de ses gradients : avec un learning rate 500 fois plus petit, il arrive au même résultat. Sur ce modèle simple, il n'apporte rien de plus ; son intérêt est que la valeur 10⁻³ convient à la plupart des réseaux, quand celle de SGD est à chercher pour chacun.

## Passage à des réseaux profonds

Pour passer à un perceptron multicouche, on crée une matrice de poids et un vecteur de biais par couche, avec `create_simple_nn_weights`. La fonction suivante les range dans une `nn.ModuleList`, un `nn.ParameterDict` `{"W": ..., "b": ...}` par couche : ces deux conteneurs permettent à `nn.Module` de trouver tous les paramètres (`model.parameters()`).

In [ ]:
def create_mlp_weights(
  n_inputs: int, n_outputs: int, hidden_layer_sizes: tuple[int, ...] = (512,)
) -> nn.ModuleList:
  layers = nn.ModuleList()
  # La taille d'entrée d'une couche est la taille de sortie de la précédente
  last_size = n_inputs
  for size in [*hidden_layer_sizes, n_outputs]:
    W, b = create_simple_nn_weights(last_size, size)
    layers.append(nn.ParameterDict({"W": W, "b": b}))
    last_size = size
  return layers

La sortie du modèle se calcule maintenant couche par couche : chaque couche prend en entrée le résultat de la précédente.

*Complétez la méthode `forward` de `MLP`. Chaque couche cachée calcule $\tanh(XW + b)$ avec `torch.tanh` ; la dernière renvoie les log-probabilités, `torch.log_softmax` de $XW + b$, comme `SimpleNN`. Les paramètres de la couche `i` sont `self.layers[i]["W"]` et `self.layers[i]["b"]`.*

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Votre code ici

### Conseils supplémentaires

- `self.layers[:-1]` : les couches cachées ; `self.layers[-1]` : la couche de sortie
- `torch.tanh(X @ layer["W"] + layer["b"])` : une couche cachée

Attendu : `MLP()(X_train[:5])` a la forme (5, 10), et `MLP((256, 128))` a 6 paramètres (`len(list(....parameters()))`).

```python
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    for layer in self.layers[:-1]:
      X = torch.tanh(...)
    output_layer = self.layers[-1]
    return torch.log_softmax(..., dim=-1)
```

### Solution

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Couches cachées : transformation affine, puis tanh
    for layer in self.layers[:-1]:
      X = torch.tanh(X @ layer["W"] + layer["b"])
    # Couche de sortie : les log-probabilités
    output_layer = self.layers[-1]
    return torch.log_softmax(X @ output_layer["W"] + output_layer["b"], dim=-1)


print(MLP()(X_train[:5]).shape, len(list(MLP((256, 128)).parameters())))

## Entraînement du réseau profond

Une couche cachée de 512 neurones, 10 epochs, un learning rate de 0,5 :

In [ ]:
mlp = MLP()
_ = train(mlp, epochs=10, learning_rate=0.5)

*Que constatez-vous en comparant ce réseau au réseau sans couche cachée ?*

Votre réponse ici

### Solution

Le réseau à une couche cachée fait nettement mieux que le réseau sans couche cachée : environ 97,5 % de justesse en validation, contre 92,5 %, soit une erreur divisée par 3. La couche cachée et sa tanh rendent le modèle non linéaire. La justesse d'entraînement approche 99 % et s'écarte de celle de validation : le réseau commence à sur-apprendre, mais sa perte de validation baisse encore. Avec un learning rate de 0,5, les courbes ne sont pas monotones : une epoch peut faire perdre un ou deux points, que les suivantes regagnent.

C'est en partie grâce à l'initialisation des poids, de variance $1 / n_\text{inputs}$. Pour vous en convaincre, retirez la division par `n_inputs**0.5` dans `create_simple_nn_weights` et relancez cette cellule et l'entraînement : avec des poids de variance 1, les trois quarts des activations tanh sont saturées (au-delà de 0,99 en valeur absolue) dès le départ, les log-probabilités initiales sont démesurées (une perte d'environ 31 au lieu de 2,3), et le réseau atteint 99,8 % sur l'entraînement mais seulement 93 % en validation. Ce grand écart ressemble à du sur-apprentissage, mais il vient d'une mauvaise initialisation (slide « Initialisation des poids »).

## Régularisation

Le perceptron précédent sur-apprend peu : 50 000 exemples suffisent à ses 400 000 poids. Le sur-apprentissage apparaît quand le réseau est grand par rapport aux données : la cellule qui suit la solution entraîne le même perceptron sur les 1 000 premiers exemples seulement, 100 epochs, sans puis avec régularisation.

La régularisation L2 ajoute à la perte une pénalité, $\lambda \sum w^2$ : $\lambda$ fois la somme des carrés des poids, qui pousse les poids vers 0 (slide « Par pénalisation des poids »). On ne pénalise que les poids `W`, pas les biais : ils ne multiplient pas l'entrée, et les contraindre n'aide pas à généraliser.

*Complétez la fonction interne de `l2`, qui renvoie la pénalité d'un `MLP` : la somme des carrés des matrices `W` de toutes ses couches (`model.layers`), multipliée par `lambda_coefficient`.*

**Documentation utile**

- [`torch.sum`](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    pass  # Votre code ici

  return penalty

### Conseils supplémentaires

- `torch.sum(layer["W"] ** 2)` : la somme des carrés d'une matrice
- `sum(...)` de Python, sur toutes les couches de `model.layers`, pour les additionner

Attendu : pour un `MLP()` neuf, `l2(1.0)(MLP())` vaut environ 522 (512 pour la première couche, 10 pour la seconde : chacune contribue à peu près son nombre de colonnes, puisque chaque poids a une variance de $1 / n_\text{inputs}$).

```python
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    weights_norm = sum(torch.sum(...) for layer in model.layers)
    return ... * weights_norm

  return penalty
```

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    # Les poids W seulement, pas les biais
    weights_norm = sum(torch.sum(layer["W"] ** 2) for layer in model.layers)
    return lambda_coefficient * weights_norm

  return penalty


print(float(l2(1.0)(MLP())))

### Sur-apprentissage sur 1 000 exemples

Le même perceptron, avec le même learning rate de 0,5, sur les 1 000 premiers exemples d'entraînement, sans régularisation puis avec une pénalité L2 de $\lambda$ = 10⁻³ ; la validation reste faite sur les 10 000 exemples mis de côté.

In [ ]:
X_small, y_small = X_train[:1000], y_train[:1000]

print("Sans régularisation")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
)

print("Avec une pénalité L2")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
  regularization=l2(1e-3),
)

*Comparez les deux entraînements : l'écart entre entraînement et validation, la perte de validation au fil des epochs, la justesse de validation finale.*

Votre réponse ici

#### Solution

Sans régularisation, le réseau apprend par cœur les 1 000 exemples : 100 % de justesse d'entraînement dès la trentième epoch, une perte d'entraînement qui tend vers 0, et une justesse de validation bloquée vers 87 %. La perte de validation est au plus bas dans les 10 premières epochs, puis remonte sans cesse, de 0,44 à 0,60 : le réseau devient de plus en plus sûr de ses réponses, y compris des fausses. C'est le sur-apprentissage.

Avec la pénalité L2, la perte d'entraînement reste au-dessus de 0,027 (les poids ne peuvent plus grandir librement), et la perte de validation remonte beaucoup moins : elle se stabilise vers 0,47. La justesse de validation, elle, ne change pas : sur si peu de données, la régularisation limite surtout l'excès de confiance du modèle. Rien ne remplace davantage de données : le même réseau atteint 97,5 % avec les 50 000 exemples.

## Recherche d'hyper-paramètres

Le learning rate et le nombre de neurones cachés sont des hyper-paramètres : on les choisit en comparant des modèles sur l'ensemble de validation, jamais sur le test. Une recherche aléatoire tire chaque essai au hasard parmi les valeurs possibles (slide « Recherche d'hyper-paramètres : grille ou aléatoire ? »).

*Complétez `search_hyperparameters` : à chaque itération, tirez un learning rate dans `learning_rates` et un nombre de neurones cachés dans `ns_hidden_units` avec `random.choice`, entraînez un `MLP` à une couche cachée pendant `epochs` epochs, puis notez ses paramètres et sa justesse de validation finale. La fonction renvoie la liste des paramètres essayés et celle des justesses. Lequel des deux hyper-paramètres compte le plus ?*

**Documentation utile**

- [`random.choice`](https://docs.python.org/fr/3/library/random.html#random.choice)

In [ ]:
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  pass  # Votre code ici

### Conseils supplémentaires

- `random.choice(learning_rates)` : une valeur tirée au hasard
- `MLP(hidden_layer_sizes=(n_hidden_units,))` : une couche cachée de `n_hidden_units` neurones
- `train(mlp, learning_rate=..., epochs=epochs, evaluate_every=epochs)` renvoie quatre listes : la deuxième contient les justesses de validation, la dernière est celle de fin d'entraînement (`evaluate_every=epochs` : une seule évaluation, à la fin)

```python
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  params, val_accs = [], []
  for _ in range(n_iters):
    learning_rate = random.choice(...)
    n_hidden_units = ...
    mlp = MLP(hidden_layer_sizes=(...,))
    _, val_accuracies, _, _ = train(
      mlp, learning_rate=..., epochs=epochs, evaluate_every=epochs
    )
    params.append({"learning_rate": ..., "n_hidden_units": ...})
    val_accs.append(...)
  return params, val_accs
```

### Solution

In [ ]:
def search_hyperparameters(
  learning_rates: tuple[float, ...] = (0.01, 0.1, 0.5),
  ns_hidden_units: tuple[int, ...] = tuple(range(32, 513, 32)),
  n_iters: int = 5,
  epochs: int = 3,
) -> tuple[list[dict[str, float]], list[float]]:
  params, val_accs = [], []
  for _ in range(n_iters):
    learning_rate = random.choice(learning_rates)
    n_hidden_units = random.choice(ns_hidden_units)
    mlp = MLP(hidden_layer_sizes=(n_hidden_units,))
    _, val_accuracies, _, _ = train(
      mlp, learning_rate=learning_rate, epochs=epochs, evaluate_every=epochs
    )
    params.append({"learning_rate": learning_rate, "n_hidden_units": n_hidden_units})
    val_accs.append(val_accuracies[-1])
  return params, val_accs


params, val_accs = search_hyperparameters()
for p, val_acc in sorted(zip(params, val_accs), key=lambda pair: -pair[1]):
  print(
    f"lr {p['learning_rate']}, {p['n_hidden_units']} neurones : val_acc {val_acc:.4f}"
  )

Le learning rate compte bien plus que le nombre de neurones : en 3 epochs, les essais à 0,01 plafonnent vers 89 %, ceux à 0,1 vers 93 % et ceux à 0,5 vers 96 %, quelle que soit la taille de la couche cachée. Avec 5 essais, une recherche aléatoire peut manquer la meilleure valeur : le nombre d'essais est un compromis avec le temps de calcul. Et 3 epochs favorisent ce qui apprend vite, pas forcément ce qui finit le mieux.

## Évaluation finale sur l'ensemble de test

L'ensemble de test n'a servi à rien jusqu'ici : ni à l'entraînement, ni au choix des hyper-paramètres. C'est ce qui rend son évaluation fiable.

*Entraînez `mlp`, un MLP aux meilleurs hyper-paramètres trouvés, pendant 10 epochs, puis calculez sa justesse sur `X_test`. Est-elle proche de la justesse de validation ?*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `params[int(np.argmax(val_accs))]` : les paramètres de l'essai à la meilleure justesse de validation
- `MLP(hidden_layer_sizes=(n,))` et `train(mlp, learning_rate=..., epochs=10)`, avec ces valeurs
- `accuracy(y_test, mlp(X_test))`, sous `torch.no_grad()` : la justesse de test

```python
best = params[...]
mlp = MLP(hidden_layer_sizes=(best[...],))
_ = train(mlp, learning_rate=..., epochs=10)
with torch.no_grad():
  print(f"Justesse sur l'ensemble de test : {accuracy(..., ...):.4f}")
```

### Solution

In [ ]:
# Les hyper-paramètres de la meilleure justesse de validation
best = params[int(np.argmax(val_accs))]
print(best)
mlp = MLP(hidden_layer_sizes=(best["n_hidden_units"],))
_ = train(mlp, learning_rate=best["learning_rate"], epochs=10)
with torch.no_grad():
  print(f"Justesse sur l'ensemble de validation : {accuracy(y_val, mlp(X_val)):.4f}")
  print(f"Justesse sur l'ensemble de test : {accuracy(y_test, mlp(X_test)):.4f}")

La justesse de test est proche de la justesse de validation, à quelques dixièmes de point (par exemple 96,0 % contre 95,9 %). L'ensemble de test, qui n'a servi à aucun choix, donne une estimation fiable des performances du modèle sur de nouvelles données ; la validation, qui a servi à choisir les hyper-paramètres, est en général légèrement optimiste.

Avec un learning rate de 0,5, la dernière epoch n'est pas toujours la meilleure : dans cet exemple, la justesse de validation est passée de 97,8 % à la neuvième epoch à 95,9 % à la dixième. Garder les poids de la meilleure epoch de validation, c'est l'arrêt précoce (slide « Par early stopping »).

## Sauvegarder et recharger le modèle

Un entraînement coûte cher : on sauvegarde le modèle entraîné pour le réutiliser sans le réentraîner. On sauvegarde son `state_dict`, le dictionnaire de ses paramètres rangés par noms, plutôt que l'objet Python lui-même (slide « Sauvegarder et recharger un modèle »).

*Sauvegardez le `state_dict` de `mlp` dans le fichier `mlp.pt`. Créez `restored`, un nouveau `MLP` de même architecture, et affichez sa justesse de test avant et après y avoir chargé ces poids.*

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `torch.save(mlp.state_dict(), "mlp.pt")` : la sauvegarde ; les clés du `state_dict` sont les chemins d'attributs des paramètres (`layers.0.W`, `layers.0.b`…)
- `restored` doit avoir la même architecture que `mlp`, mêmes `hidden_layer_sizes` : mêmes noms de paramètres, mêmes formes
- `torch.load("mlp.pt", map_location=device)` : relit le dictionnaire, ses tenseurs placés sur `device` ; `restored.load_state_dict(...)` copie ses valeurs dans les paramètres de `restored`
- `accuracy(y_test, restored(X_test))`, sous `torch.no_grad()` : la justesse de test

Attendu : environ 0,1 avant le chargement, exactement la justesse de `mlp` après.

```python
torch.save(..., "mlp.pt")
restored = MLP(hidden_layer_sizes=(...,))
with torch.no_grad():
  print(accuracy(y_test, restored(X_test)))
restored.load_state_dict(torch.load(..., map_location=...))
with torch.no_grad():
  print(accuracy(..., ...))
```

### Solution

In [ ]:
torch.save(mlp.state_dict(), "mlp.pt")
# Les clés : le chemin d'attributs de chaque paramètre
print(list(mlp.state_dict().keys()))

# Nouveaux poids aléatoires, même architecture que mlp
restored = MLP(hidden_layer_sizes=(best["n_hidden_units"],))
with torch.no_grad():
  print(f"Avant chargement : {accuracy(y_test, restored(X_test)):.4f}")
restored.load_state_dict(torch.load("mlp.pt", map_location=device))
with torch.no_grad():
  print(f"Après chargement : {accuracy(y_test, restored(X_test)):.4f}")
  print(f"Modèle sauvegardé : {accuracy(y_test, mlp(X_test)):.4f}")

Avant le chargement, `restored` répond presque au hasard (de l'ordre de 10 %, ses poids sont aléatoires). Après, il a exactement la justesse de `mlp` : `load_state_dict` a copié chaque tenseur dans le paramètre de même nom (`layers.0.W`, `layers.0.b`, `layers.1.W`, `layers.1.b`). D'où l'importance d'une architecture identique : un `MLP` d'une autre taille aurait des paramètres de formes différentes, et le chargement échouerait. `map_location` place les tenseurs lus sur `device` : sans lui, des poids sauvegardés sur un GPU ne se chargent pas sur une machine qui n'en a pas.

Le fichier ne contient que des valeurs, pas le code du modèle : pour le recharger, il faut la classe `MLP`. Notre `MLP` n'a ni dropout ni normalisation de batch ; pour un modèle qui en a, on appelle `restored.eval()` avant de prédire. Et pour reprendre un entraînement, on sauvegarde aussi `optimizer.state_dict()` et le numéro d'epoch.